# FLUX 3 Image Klein on Kaggle (Wan2GP)

Single T4 setup. Heavy weights go in `/tmp`.

Uses the step-distilled INT8 ConvRot checkpoint from `DeepBeepMeep/Flux3` (5.6B, no CFG, 4-16 steps).

## 1. Install Wan2GP

In [ ]:
%cd /kaggle/working
import os
if not os.path.exists('/kaggle/working/Wan2GP'):
    !git clone https://github.com/deepbeepmeep/Wan2GP.git
else:
    !git -C /kaggle/working/Wan2GP pull
%cd /kaggle/working/Wan2GP

## 2. Install requirements

In [ ]:
!pip install -r requirements.txt

## 3. Download heavy models to `/tmp`

Only the transformer and text encoder. VAE (`flux3/flux2-vae.safetensors`, 336 MB) is small; let Wan2GP download it.

T4 has no BF16 or NVFP4 support, so INT8 ConvRot is the right pick here.

### Transformer

In [ ]:
from huggingface_hub import hf_hub_download

REPO = 'DeepBeepMeep/Flux3'
LOCAL = '/tmp/flux3_klein'

# Step-distilled INT8 (5.83 GB). Full / base version: 'flux3_image_klein_int8_convrot.safetensors'
TRANSFORMER = 'flux3_image_klein_sd_int8_convrot.safetensors'

base = hf_hub_download(repo_id=REPO, filename=TRANSFORMER, local_dir=LOCAL)
print('Transformer:', base)
!ls -lh /tmp/flux3_klein

### Text encoder (Qwen3-VL-4B, INT8)

In [ ]:
ENC_DIR = 'Qwen3-VL-4B-Instruct'
for f in [
    'qwen3vl_4b_int8_convrot.safetensors',
    'config.json',
    'tokenizer.json',
    'tokenizer_config.json',
    'chat_template.jinja',
]:
    p = hf_hub_download(repo_id=REPO, filename=f'{ENC_DIR}/{f}', local_dir=LOCAL)
    print('Got:', p)
!du -sh /tmp/flux3_klein

## 4. Launch Wan2GP

Profile 5 is the low-VRAM profile. Only one T4 will be used.

In [ ]:
%cd /kaggle/working/Wan2GP
!python wgp.py --share --profile 5

## 5. Select the model in Wan2GP

Recent Wan2GP versions should list **Flux 3 Image Klein** in the model dropdown. If it's listed, pick it and try it first, since it may auto-download everything.

If not, or if you want to use the files in `/tmp`, add them as a finetune: pick the closest Flux 3 / Flux 2 Klein entry, open Finetune Creator / Editor, and point these fields at the `/tmp` files.

### Main checkpoint
```
/tmp/flux3_klein/flux3_image_klein_sd_int8_convrot.safetensors
```

### Text encoder checkpoint
```
/tmp/flux3_klein/Qwen3-VL-4B-Instruct/qwen3vl_4b_int8_convrot.safetensors
```

Keep the default Flux 3 / Flux 2 VAE from the source model.

## 6. Settings

| Setting | Value |
|---|---|
| Resolution | `1024x1024` |
| Steps | `8` for speed, `16` for best quality (official distilled value) |
| Guidance (CFG) | `1` (distilled, guidance-free) |
| Acceleration LoRA | none |
| KV Cache | Off |
| Batch | `1` |

Native 4K is supported, but stay at 1024x1024 on a T4 until you know it fits.

For image edit, connect reference images and refer to them as `image 1`, `image 2` in the prompt (up to 10).

## 7. Test prompt

**Prompt**
```
A cute orange cat sitting on a wooden table beside a small cup of coffee, warm morning sunlight coming through a window, realistic photography.
```

Negative prompts do nothing at CFG 1, so leave it empty.

## License

FLUX 3 Image Klein weights use the FLUX Kommunity License: free commercial use of outputs only if your gross annual revenue (including affiliates) is **under $5M**. Non-commercial and research use is free. Check the license before using outputs commercially.